# Transcript & Caption Generation

Generate **whisper-large-v3** transcripts and **InternVL2.5-8B** visual captions for every video/audio segment pair. Outputs:

```
outputs/model_embeddings/text/{seg}s/{captions,transcripts}.json
```

Audio captions and the LLM rewrite step are handled in `audiocaption.ipynb` (audiocaption env).

**Kernel:** `avtransformer`

## loading

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from paths import DATA, OUTPUTS, MODELS

import gc
import json
import os
import re
from pathlib import Path

import librosa
import numpy as np
import torch
from natsort import natsorted
from PIL import Image
from decord import VideoReader, cpu as dcpu
from torchvision.transforms.functional import InterpolationMode
import torchvision.transforms as T
from transformers import (
    AutoModel, AutoTokenizer,
    WhisperForConditionalGeneration, WhisperProcessor,
    BitsAndBytesConfig,
)

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# -- CONFIG -------------------------------------------------------------------
EXTERNAL_HF_CACHE  = str(MODELS / "hub")
WHISPER_LOCAL_PATH = (
    str(MODELS / "manual/models--openai--whisper-large-v3")
)
VL_MODEL_ID    = "OpenGVLab/InternVL2_5-8B"       # video captioner (4-bit, ~4.5 GB)
GEMMA_MODEL_ID = "google/gemma-4-E4B-it"           # audio captioner (~9 GB bfloat16)

VL_NUM_SEGMENTS    = 8      # frames per video segment; auto-halved on OOM
WHISPER_BATCH_SIZE = 16
USE_VAD = True              # gate Whisper with Silero VAD; False = transcribe everything

# BIN_SEC: window duration; SKIP_SEC: stride (= BIN_SEC → non-overlapping)
BIN_SEC  = 5.0
SKIP_SEC = 5.0

# Each tuple: (bin_sec, skip_sec) — output dir: bin{N}s_skip{N}s
SEGMENT_DURATIONS = [(5.0, 5.0), (10.0, 10.0)]

DATA_BASE   = DATA / "segmented_stimulus/filtered"
OUTPUT_BASE = OUTPUTS / "model_embeddings"
TEXT_OUT    = OUTPUT_BASE / "text"

VL_CAPTION_PROMPT = (
    "Describe in one or two concise sentences what is visually happening in this video clip."
)
AUDIO_CAPTION_PROMPT = (
    "In one sentence, describe the sounds in this audio clip: "
    "acoustic events, music style and mood, speech presence, and ambient environment."
)
# -----------------------------------------------------------------------------


def find_chunk_pairs(data_base: Path, bin_sec: float, skip_sec: float = None) -> list[tuple[Path, Path]]:
    """Return sorted (av_mp4, av_mp4) pairs from combined AV chunk dirs.

    Looks for {stem}_av_chunks_{dur}s or {stem}_av_chunks_{dur}s_skip{skip}s.
    Both tuple elements point to the same .mp4 (video + audio from one file).
    """
    if skip_sec is None:
        skip_sec = bin_sec
    dur_int  = int(bin_sec)
    skip_int = int(skip_sec)
    chunk_suffix = (f"_av_chunks_{dur_int}s" if skip_int == dur_int
                    else f"_av_chunks_{dur_int}s_skip{skip_int}s")
    pairs = []
    for vid_subdir in natsorted(data_base.glob("Video*")):
        m = re.search(r"Video(\d+)$", vid_subdir.name)
        if not m:
            continue
        chunk_dir = vid_subdir / f"{vid_subdir.name}{chunk_suffix}"
        if not chunk_dir.exists():
            continue
        for av in natsorted(chunk_dir.glob("*_part_*.mp4")):
            pairs.append((av, av))
    return pairs


print("Chunk pair counts per segment duration:")
for _b, _s in SEGMENT_DURATIONS:
    _p = find_chunk_pairs(DATA_BASE, _b, _s)
    print(f"  bin{int(_b)}s_skip{int(_s)}s -> {len(_p):4d} pairs")

## Model Downloads (run once)

Download **InternVL2.5-8B** and **Gemma 4 E4B** to the external HDD HF cache via the cells below.

**Whisper-large-v3** is already on the HDD (`hf_models/manual/`). Silero VAD downloads via torch.hub on first run.

> **Note:** Gemma 4 requires accepting the Google license. Run `huggingface-cli login` and accept the license at https://huggingface.co/google/gemma-4-E4B-it before the first download.

In [ ]:
from huggingface_hub import snapshot_download
import shutil


def _hf_weights_present(ext_model_dir: Path) -> bool:
    '''True when at least one non-index safetensors shard is present in any snapshot.'''
    snap_dir = ext_model_dir / "snapshots"
    if not snap_dir.exists():
        return False
    for snap in snap_dir.iterdir():
        if snap.is_dir() and any(
            f.suffix == ".safetensors" and "index" not in f.name
            for f in snap.iterdir()
        ):
            return True
    return False


_org, _name  = VL_MODEL_ID.split("/")
_ext_vl_dir  = Path(EXTERNAL_HF_CACHE) / f"models--{_org}--{_name}"
_default_hub = Path.home() / ".cache" / "huggingface" / "hub"
_sym_vl      = _default_hub / f"models--{_org}--{_name}"

if not _hf_weights_present(_ext_vl_dir):
    print(f"Downloading {VL_MODEL_ID} to external HDD ...")
    snapshot_download(VL_MODEL_ID, cache_dir=EXTERNAL_HF_CACHE,
                      ignore_patterns=["*.msgpack", "flax_model*"])
    print(f"Downloaded: {_ext_vl_dir}")
else:
    print(f"Weights already on HDD: {_ext_vl_dir}")

if _sym_vl.exists() and not _sym_vl.is_symlink():
    shutil.rmtree(_sym_vl)
if not _sym_vl.exists():
    _sym_vl.symlink_to(_ext_vl_dir)
    print(f"Symlink: {_sym_vl} -> {_ext_vl_dir}")
else:
    print(f"Symlink exists: {_sym_vl}")
print(f"Weights present: {_hf_weights_present(_ext_vl_dir)}")


In [ ]:
_gm_org, _gm_name = GEMMA_MODEL_ID.split("/")
_ext_gm_dir = Path(EXTERNAL_HF_CACHE) / f"models--{_gm_org}--{_gm_name}"
_default_hub = Path.home() / ".cache" / "huggingface" / "hub"
_sym_gm      = _default_hub / f"models--{_gm_org}--{_gm_name}"

if not _hf_weights_present(_ext_gm_dir):
    print(f"Downloading {GEMMA_MODEL_ID} to external HDD ...")
    snapshot_download(GEMMA_MODEL_ID, cache_dir=EXTERNAL_HF_CACHE,
                      ignore_patterns=["*.msgpack", "flax_model*"])
    print(f"Downloaded: {_ext_gm_dir}")
else:
    print(f"Weights already on HDD: {_ext_gm_dir}")

if _sym_gm.exists() and not _sym_gm.is_symlink():
    shutil.rmtree(_sym_gm)
if not _sym_gm.exists():
    _sym_gm.symlink_to(_ext_gm_dir)
    print(f"Symlink: {_sym_gm} -> {_ext_gm_dir}")
else:
    print(f"Symlink exists: {_sym_gm}")
print(f"Weights present: {_hf_weights_present(_ext_gm_dir)}")

## Transcripts — whisper-large-v3 + Silero VAD

Silero VAD runs first on every segment. Only segments where speech is detected
are passed to Whisper — music/SFX/silent segments return an empty string, avoiding
Whisper hallucinations. Set `USE_VAD = False` in CONFIG to transcribe everything.


In [ ]:
torch.cuda.empty_cache(); gc.collect()
print("Loading Whisper ...")

whisper_proc  = WhisperProcessor.from_pretrained(WHISPER_LOCAL_PATH, local_files_only=True)
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_LOCAL_PATH, local_files_only=True, torch_dtype=torch.float16,
).to(DEVICE)
whisper_model.eval()
forced_decoder_ids = whisper_proc.get_decoder_prompt_ids(language="en", task="transcribe")
print("Whisper loaded.")

# Silero VAD — cached to ~/.cache/torch/hub after first download
_vad_model, _vad_utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad', model='silero_vad',
    force_reload=False, onnx=False, trust_repo=True,
)
(_get_speech_ts, _, _vad_read_audio, _, _) = _vad_utils
_vad_model = _vad_model.to(DEVICE)
print("Silero VAD loaded.")


def _has_speech(wav_path: Path) -> bool:
    wav = _vad_read_audio(str(wav_path), sampling_rate=16000).to(DEVICE)
    ts  = _get_speech_ts(wav, _vad_model, sampling_rate=16000,
                         threshold=0.5, min_silence_duration_ms=300)
    return len(ts) > 0


@torch.no_grad()
def generate_transcripts(aud_paths: list, batch_size: int) -> list:
    '''Whisper transcription with optional Silero VAD gating (controlled by USE_VAD).'''
    if USE_VAD:
        print("  Running Silero VAD ...", end=" ")
        speech_flags = [_has_speech(p) for p in aud_paths]
        print(f"{sum(speech_flags)}/{len(aud_paths)} segments have speech")
    else:
        speech_flags = [True] * len(aud_paths)

    results  = []
    cur_size = batch_size
    i = 0
    while i < len(aud_paths):
        if not speech_flags[i]:
            results.append("")
            i += 1
            continue
        # Collect consecutive speech segments up to cur_size
        batch = []
        j = i
        while j < len(aud_paths) and len(batch) < cur_size and speech_flags[j]:
            batch.append(aud_paths[j])
            j += 1
        try:
            audios = [librosa.load(p, sr=16000, mono=True)[0] for p in batch]
            inputs = whisper_proc(audios, sampling_rate=16000, return_tensors="pt", padding=True)
            feats  = inputs.input_features.to(DEVICE, dtype=torch.float16)
            ids    = whisper_model.generate(feats, forced_decoder_ids=forced_decoder_ids)
            texts  = whisper_proc.batch_decode(ids, skip_special_tokens=True)
            results.extend([t.strip() for t in texts])
            i += len(batch)
            print(f"    {i}/{len(aud_paths)} transcribed (batch={cur_size})", end="\r")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            if cur_size == 1: raise
            cur_size = max(1, cur_size // 2)
            print(f"\n    OOM — reducing batch size to {cur_size}")
    print()
    return results

In [ ]:
for bin_sec, skip_sec in SEGMENT_DURATIONS:
    out_dir  = TEXT_OUT / f"bin{int(bin_sec)}s_skip{int(skip_sec)}s"
    out_path = out_dir / "transcripts.json"

    if out_path.exists():
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- transcripts already saved, skipping")
        continue

    pairs = find_chunk_pairs(DATA_BASE, bin_sec, skip_sec)
    if not pairs:
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- no pairs found, skipping")
        continue

    print(f"\nbin{int(bin_sec)}s_skip{int(skip_sec)}s -- transcribing {len(pairs)} segments ...")
    transcripts = generate_transcripts([p[1] for p in pairs], WHISPER_BATCH_SIZE)

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(transcripts, ensure_ascii=False, indent=2))
    print(f"  Saved: {out_path}  ({len(transcripts)} entries)")

In [ ]:
del whisper_model, whisper_proc
torch.cuda.empty_cache()
gc.collect()
print("Whisper unloaded.")


## Audio Captions — Gemma 4 E4B

`google/gemma-4-E4B-it` — Google's Gemma 4 multimodal model with a built-in audio encoder (~300M params).
Loaded in `dtype="auto"` (bfloat16, ~9 GB VRAM). GPU is cleared before load and after unload.

Output: `outputs/model_embeddings/text/{seg}s/audio_captions.json`

In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor,BitsAndBytesConfig

torch.cuda.empty_cache(); gc.collect()
print("Loading Gemma 4 E4B ...")
try:
    del gemma_model
    del gemma_proc
except NameError:
    pass

torch.cuda.empty_cache()
gc.collect()
gemma_proc  = AutoProcessor.from_pretrained(GEMMA_MODEL_ID)
gemma_model = AutoModelForMultimodalLM.from_pretrained(
    GEMMA_MODEL_ID,
    dtype=torch.bfloat16,
    quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_skip_modules=["audio_tower", "embed_audio", "lm_head"]
),device_map={"": 0}
)
gemma_model.eval()
print("Gemma 4 E4B loaded.")


@torch.no_grad()
def generate_audio_captions(aud_paths: list) -> list:
    results = []
    for i, aud_path in enumerate(aud_paths):
        messages = [{
            "role": "user",
            "content": [
                {"type": "text",  "text":  AUDIO_CAPTION_PROMPT},
                {"type": "audio", "audio": str(aud_path)},
            ],
        }]
        inputs = gemma_proc.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
        ).to(DEVICE)
        input_len = inputs["input_ids"].shape[-1]
        out = gemma_model.generate(**inputs, max_new_tokens=128, do_sample=False)
        raw = gemma_proc.decode(out[0][input_len:], skip_special_tokens=False)
        try:
            caption = gemma_proc.parse_response(raw).strip()
        except Exception:
            caption = gemma_proc.decode(out[0][input_len:], skip_special_tokens=True).strip()
        results.append(caption)
        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{len(aud_paths)} audio-captioned", end="\r")
    print()
    return results

In [ ]:
for bin_sec, skip_sec in SEGMENT_DURATIONS:
    out_dir  = TEXT_OUT / f"bin{int(bin_sec)}s_skip{int(skip_sec)}s"
    out_path = out_dir / "audio_captions.json"

    if out_path.exists():
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- audio_captions.json already exists, skipping")
        continue

    pairs = find_chunk_pairs(DATA_BASE, bin_sec, skip_sec)
    if not pairs:
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- no pairs found, skipping")
        continue

    print(f"\nbin{int(bin_sec)}s_skip{int(skip_sec)}s -- audio-captioning {len(pairs)} segments ...")
    audio_captions = generate_audio_captions([p[1] for p in pairs])

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(audio_captions, ensure_ascii=False, indent=2))
    print(f"  Saved: {out_path}  ({len(audio_captions)} entries)")
    if audio_captions:
        print(f"  Sample [0]: {audio_captions[0][:120]}")

In [ ]:
del gemma_model, gemma_proc
torch.cuda.empty_cache()
gc.collect()
print("Gemma 4 E4B unloaded.")

## Video Captions — InternVL2.5-8B (4-bit)

`OpenGVLab/InternVL2_5-8B` — vision-language model, loaded 4-bit NF4 (~4.5 GB VRAM).

Output: `outputs/model_embeddings/text/{seg}s/captions.json`

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def _vl_transform(input_size: int = 448) -> T.Compose:
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def load_video_frames(
    video_path, num_segments: int = VL_NUM_SEGMENTS, input_size: int = 448
) -> tuple:
    vr  = VideoReader(str(video_path), ctx=dcpu(0), num_threads=1)
    n   = len(vr)
    seg = n / num_segments
    indices = [int(seg / 2 + seg * i) for i in range(num_segments)]
    tfm     = _vl_transform(input_size)
    frames  = [tfm(Image.fromarray(vr[fi].asnumpy()).convert("RGB")) for fi in indices]
    return torch.stack(frames), [1] * num_segments


torch.cuda.empty_cache(); gc.collect()
print("Loading InternVL2.5-8B ...")


vl_model = AutoModel.from_pretrained(
    VL_MODEL_ID,
    torch_dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    ),
    device_map="auto",
    low_cpu_mem_usage=True,
    use_flash_attn=False,
    trust_remote_code=True
).eval()

vl_tokenizer = AutoTokenizer.from_pretrained(
    VL_MODEL_ID, trust_remote_code=True, use_fast=False
)
_vl_device = next(p for p in vl_model.parameters() if p.device.type == "cuda").device
print(f"InternVL2.5-8B loaded  device={_vl_device}")


@torch.no_grad()
def generate_captions(vid_paths: list) -> list:
    results  = []
    gen_cfg  = dict(max_new_tokens=128, do_sample=False)
    cur_segs = VL_NUM_SEGMENTS
    for i, vid_path in enumerate(vid_paths):
        while True:
            try:
                pv, npl = load_video_frames(vid_path, num_segments=cur_segs)
                pv = pv.to(torch.bfloat16).to(_vl_device)
                response = vl_model.chat(
                    vl_tokenizer, pv, VL_CAPTION_PROMPT, gen_cfg,
                    num_patches_list=npl,
                )
                results.append(response.strip())
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); gc.collect()
                if cur_segs <= 1:
                    raise
                cur_segs = max(1, cur_segs // 2)
                print(f"\n  OOM at [{i}] — reducing frames to {cur_segs}")
        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{len(vid_paths)} captioned (frames={cur_segs})", end="\r")
    print()
    return results

In [ ]:
for bin_sec, skip_sec in SEGMENT_DURATIONS:
    out_dir  = TEXT_OUT / f"bin{int(bin_sec)}s_skip{int(skip_sec)}s"
    out_path = out_dir / "captions.json"

    if out_path.exists():
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- captions already saved, skipping")
        continue

    pairs = find_chunk_pairs(DATA_BASE, bin_sec, skip_sec)
    if not pairs:
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- no pairs found, skipping")
        continue

    print(f"\nbin{int(bin_sec)}s_skip{int(skip_sec)}s -- captioning {len(pairs)} segments ...")
    captions = generate_captions([p[0] for p in pairs])

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(captions, ensure_ascii=False, indent=2))
    print(f"  Saved: {out_path}  ({len(captions)} entries)")
    if captions:
        print(f"  Sample [0]: {captions[0][:120]}")

In [ ]:
del vl_model, vl_tokenizer
torch.cuda.empty_cache()
gc.collect()
print("InternVL2.5-8B unloaded.")
